# BEATs: positive-source diversity and full fine-tuning

Run all cells with a **GPU runtime**. This notebook starts from the downloaded
**222 original distributions (39.011 GB)**. It does **not** require a previous
trained model, feature archive or completed notebook. It also acquires the
6.26 GB FSD50K evaluation audio and bounded WSU/AuDroK source material.

The complete cohort has **30,238 canonical crops**. For each of two whole-source
holdouts (WSU and AuDroK), compare current versus broader positive training,
first with frozen BEATs and then with all **90.3 million** encoder parameters
trainable. Every neural arm visits every selected training crop for three
epochs. Model/threshold selection excludes the held source. This is development
validation on previously examined collections, not a new generalization test.
Publisher presence labels remain weak audibility labels.

Original archives are read only. Verified canonical audio, model parameters,
optimizer recovery checkpoints and results persist in the separate Drive project.
Training reads verified local binary copies for speed. Use **one active runtime
per project**; CPU and GPU studies have separate directories and recompute their
own matched frozen controls. Keep the same GPU type when resuming a neural arm.
The pipeline pauses at a batch checkpoint after a cumulative 44-hour budget.
It never replaces the study with a small sample. No automatic future execution
or notification is configured by this notebook.


In [ ]:
ARCHIVES_DRIVE_PATH = '/content/drive/MyDrive/drone-detection-v2/corpus-intake-v3/source-archives'
PROJECT_DRIVE_PATH = '/content/drive/MyDrive/drone-beats-adaptation-v1'
REPO_URL = 'https://github.com/StoneDerek/drone-detection.git'
CODE_COMMIT = '2a4de5723cf755ab3a9187c8c6027030cc616027'
DEVICE = 'cuda'
THREADS = 1
FEATURE_SHARDS = 1  # Bootstrap contract; training has a single writer.
if DEVICE != 'cuda':
    raise ValueError('Select a GPU runtime for this notebook; CPU is a separate study')


In [ ]:
import hashlib
import json
import os
import shutil
import subprocess
import sys
import urllib.request
from pathlib import Path

from google.colab import drive

if DEVICE not in {'cpu', 'cuda'} or FEATURE_SHARDS not in {1, 2, 3, 4}:
    raise ValueError('Choose an explicit supported device and worker count')
if DEVICE == 'cuda':
    subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv'], check=True)
drive.mount('/content/drive')
MOUNT = Path('/content/drive').resolve()
if not os.path.ismount(MOUNT):
    raise RuntimeError('Drive must be mounted for durable results')
PROJECT = Path(PROJECT_DRIVE_PATH).resolve()
ARCHIVES = Path(ARCHIVES_DRIVE_PATH).resolve()
if not PROJECT.is_relative_to(MOUNT) or PROJECT == MOUNT or not ARCHIVES.is_relative_to(MOUNT):
    raise ValueError('Choose the expected mounted Drive paths')
if PROJECT == ARCHIVES or PROJECT.is_relative_to(ARCHIVES) or ARCHIVES.is_relative_to(PROJECT):
    raise ValueError('Keep the result project separate from the input archives')
PROJECT.mkdir(parents=True, exist_ok=True)
if (PROJECT / 'bootstrap.json').exists() and json.loads((PROJECT / 'bootstrap.json').read_text()).get('revision') != CODE_COMMIT:
    raise ValueError('This project has another code pin; preserve it and use a new project folder')
print('Input archives:', ARCHIVES)
print('Durable results:', PROJECT)
print('Local runtime disk:', shutil.disk_usage('/content'))


In [ ]:
"""Standard-library-only bootstrap, embedded verbatim in the Colab notebook."""

import hashlib
import json
import os
import re
import subprocess
import uuid
from pathlib import Path


def git_output(checkout, *arguments):
    return subprocess.check_output(['git', '-C', str(checkout), *arguments], text=True).strip()


def file_hash(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def pin_checkout(project, checkout, repo_url, first_ref='main'):
    """Resolve once, then reuse that exact revision after runtime replacement.

    One active runtime per project. Refuse dirty or unrelated local checkouts.
    The saved lock hash is also checked; a pin is never silently refreshed.
    """
    project, checkout = Path(project), Path(checkout)
    project.mkdir(parents=True, exist_ok=True)
    pin_path = project / 'bootstrap.json'
    pin = json.loads(pin_path.read_text()) if pin_path.exists() else None
    if pin is not None and (pin.get('schema_version') != 1
                            or pin.get('repo_url') != repo_url
                            or not re.fullmatch(r'[0-9a-f]{40}', pin.get('revision', ''))
                            or not re.fullmatch(r'[0-9a-f]{64}', pin.get('runtime_lock_sha256', ''))):
        raise ValueError('Invalid/different project pin; choose a new project folder deliberately')
    if checkout.exists() and any(checkout.iterdir()):
        if not (checkout / '.git').exists():
            raise ValueError('Checkout folder contains unrelated files')
        if git_output(checkout, 'remote', 'get-url', 'origin') != repo_url:
            raise ValueError('Checkout has a different repository origin')
        if git_output(checkout, 'status', '--porcelain'):
            raise ValueError('Local checkout changed; preserve those changes before continuing')
    else:
        checkout.mkdir(parents=True, exist_ok=True)
        subprocess.run(['git', 'init', '--quiet', str(checkout)], check=True)
        subprocess.run(['git', '-C', str(checkout), 'remote', 'add', 'origin', repo_url], check=True)
    target = pin['revision'] if pin else first_ref
    subprocess.run(['git', '-C', str(checkout), 'fetch', '--quiet', '--depth', '1', 'origin', target], check=True)
    revision = git_output(checkout, 'rev-parse', 'FETCH_HEAD')
    if pin and revision != pin['revision']:
        raise ValueError('Fetched revision differs from saved project pin')
    subprocess.run(['git', '-C', str(checkout), 'checkout', '--quiet', '--detach', revision], check=True)
    candidate = {'schema_version': 1, 'repo_url': repo_url, 'revision': revision,
                 'runtime_lock_sha256': file_hash(checkout / 'notebooks/runtime/uv.lock')}
    if pin is not None and pin != candidate:
        raise ValueError('Pinned environment lock changed')
    if pin is None:
        payload = json.dumps(candidate, indent=2, sort_keys=True).encode()
        temporary = pin_path.with_name('bootstrap.partial-' + uuid.uuid4().hex)
        temporary.write_bytes(payload)
        if pin_path.exists() and json.loads(pin_path.read_text()) != candidate:
            raise ValueError('Another bootstrap wrote a different pin; use one active runtime')
        os.replace(temporary, pin_path)
        if pin_path.read_bytes() != payload:
            raise ValueError('Project pin read-back failed')
    return candidate


REPO = Path('/content/drone-beats-adaptation-code')
PIN = pin_checkout(PROJECT, REPO, REPO_URL, CODE_COMMIT)
assert PIN['revision'] == CODE_COMMIT
print('Exact implementation:', PIN['revision'])
print('Runtime lock SHA-256:', PIN['runtime_lock_sha256'])


In [ ]:
UV_DIR = Path('/content/drone-corpus-bootstrap')
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '--upgrade', '--target', str(UV_DIR), 'uv==0.13.5'], check=True)
UV_ENV = dict(os.environ, PYTHONPATH=str(UV_DIR), UV_CACHE_DIR='/content/drone-corpus-uv-cache', UV_PYTHON_INSTALL_DIR='/content/drone-corpus-python')
subprocess.run([sys.executable, '-m', 'uv', 'sync', '--project', str(REPO / 'notebooks/runtime'), '--frozen', '--python', '3.12',
                '--extra', 'gpu' if DEVICE == 'cuda' else 'cpu', '--extra', 'probe'], env=UV_ENV, check=True)
PYTHON = REPO / 'notebooks/runtime/.venv/bin/python'
JOB_ENV = dict(os.environ, PYTHONPATH=str(REPO / 'src'), OPENBLAS_NUM_THREADS='1', OMP_NUM_THREADS='1', CUBLAS_WORKSPACE_CONFIG=':4096:8')
check = "import torch; print('Torch:',torch.__version__); "
check += "assert torch.cuda.is_available(); print('GPU:',torch.cuda.get_device_name(0))" if DEVICE == 'cuda' else "print('Explicit CPU execution; full corpus will still be processed')"
subprocess.run([str(PYTHON), '-c', check], cwd=REPO, env=JOB_ENV, check=True)

def run_script(name, *arguments):
    command = [str(PYTHON), str(REPO / 'scripts' / name), *[str(value) for value in arguments]]
    subprocess.run(command, cwd=REPO, env=JOB_ENV, check=True)


## Verify the original distributions and prepare the full canonical corpus

The first run independently checks publisher digests for all 222 files. It uses
a local symlink view to add verification receipts without changing the originals.
Completed preparation is reused after a runtime replacement. Missing or
conflicting inputs fail explicitly. Archive verification and audio preparation
take time before training begins; no old embeddings are needed.


In [ ]:
LOCAL = Path('/content/drone-beats-adaptation-inputs')
CANONICAL = PROJECT / 'canonical-inputs'
LOCAL.mkdir(parents=True, exist_ok=True)
CANONICAL.mkdir(parents=True, exist_ok=True)

def durable_link(link, target):
    target.mkdir(parents=True, exist_ok=True)
    link.parent.mkdir(parents=True, exist_ok=True)
    if link.is_symlink():
        if link.resolve() != target.resolve():
            raise ValueError('Existing local input link points elsewhere: ' + str(link))
    elif link.exists():
        raise ValueError('Preserve the existing local input folder: ' + str(link))
    else:
        link.symlink_to(target, target_is_directory=True)

PREVIOUS = LOCAL / 'milestone-v1'
FIELD = LOCAL / 'confuser-study-v1'
durable_link(PREVIOUS / 'prepared', CANONICAL / 'original-prepared')
durable_link(FIELD / 'prepared', CANONICAL / 'field-prepared')
durable_link(FIELD / 'archives', CANONICAL / 'field-archives')
URBAN = LOCAL / 'UrbanSound8K.csv'
if not URBAN.exists():
    url = 'https://raw.githubusercontent.com/MOULIK-RAZDAN/AUDIO-CLASSIFICATION/main/UrbanSound8K.csv'
    with urllib.request.urlopen(url, timeout=60) as response:
        URBAN.write_bytes(response.read())
if hashlib.sha256(URBAN.read_bytes()).hexdigest() != 'b207f7a53f9e5eb02a5dd71d58fb8b66c88c24c0637b926dc72d9d400a6a2354':
    raise ValueError('Pinned UrbanSound metadata changed')
if not (PREVIOUS / 'prepared/preparation-complete.json').exists():
    VERIFIED_ARCHIVES = LOCAL / 'verified-source-archives'
    run_script('verify_adaptation_archives.py', '--archives', ARCHIVES, '--output', VERIFIED_ARCHIVES)
    run_script('prepare_corpus_study.py', '--archives', VERIFIED_ARCHIVES,
               '--output', PREVIOUS / 'prepared', '--urban-csv', URBAN)
    run_script('finalize_corpus_study.py', '--prepared', PREVIOUS / 'prepared', '--archives', VERIFIED_ARCHIVES)
if not (FIELD / 'prepared/preparation-complete.json').exists():
    run_script('acquire_confuser_study.py', '--work', FIELD, '--metadata-only')
    run_script('prepare_confuser_study.py', 'plan', '--work', FIELD, '--previous', PREVIOUS,
               '--urban-metadata', URBAN)
    run_script('acquire_confuser_study.py', '--work', FIELD)
    for stage in ['wsu', 'audrok', 'fsd', 'seal']:
        run_script('prepare_confuser_study.py', stage, '--work', FIELD)
completion = json.loads((FIELD / 'prepared/preparation-complete.json').read_text())
manifest = FIELD / 'prepared/manifest.jsonl'
if hashlib.sha256(manifest.read_bytes()).hexdigest() != completion['manifest_sha256']:
    raise ValueError('Canonical preparation receipt no longer matches its manifest')
print('Complete canonical corpus:', completion)


In [ ]:
STAGED_CORPUS = LOCAL / 'training-inputs'
run_script('stage_adaptation_audio.py', '--manifest', manifest, '--output', STAGED_CORPUS)
DURABLE_ASSETS = PROJECT / 'model-assets/beats'
run_script('fetch_beats.py', '--output', DURABLE_ASSETS)
ASSETS = LOCAL / 'model-assets/beats'
shutil.copytree(DURABLE_ASSETS, ASSETS, dirs_exist_ok=True)
WORK = PROJECT / DEVICE / 'study'
REPORT = WORK / 'report'
run_script('run_beats_adaptation.py', 'plan', '--corpus', STAGED_CORPUS,
           '--work', WORK, '--report', REPORT)
shutil.copyfile(REPO / 'reports/beats-adaptation-v1/PROTOCOL.md', REPORT / 'PROTOCOL.md')
print('Whole-source folds and three-epoch policy frozen:', WORK / 'plan.json')


## Run the matched frozen controls and all four neural arms

The launcher first extracts native BEATs features for the complete cohort, then
fits four frozen controls, then trains four full-backbone models. Progress and
recovery state are saved continuously. A runtime restart resumes the optimizer
and exact next batch; completed stages are verified and reused. Partial feature
extraction is repeated if interrupted before its atomic matrix checkpoint.

The cell displays the current stage and recent log entries once per minute.
Interrupting the cell requests a cooperative pause; allow the child time to
finish its batch and write recovery state. Do not start another runtime writer.
A budget pause requires a deliberate decision before extending the compute
budget; rerunning the notebook does not silently grant more time.


In [ ]:
import time

pause = WORK / 'pause-requested.json'
if pause.exists():
    raise RuntimeError('Study is paused. Preserve the checkpoint and review ' + str(pause))
command = [str(PYTHON), '-u', str(REPO / 'scripts/launch_beats_adaptation.py'),
           '--work', str(WORK), '--assets', str(ASSETS), '--python', str(PYTHON),
           '--device', DEVICE, '--threads', str(THREADS), '--max-wall-hours', '44']
supervisor_log = WORK / 'supervisor.log'
with supervisor_log.open('a') as log:
    process = subprocess.Popen(command, cwd=REPO, env=JOB_ENV, stdout=log,
                               stderr=subprocess.STDOUT, start_new_session=True)
    last_report = 0
    try:
        while process.poll() is None:
            if time.monotonic() - last_report >= 60:
                state_file = WORK / 'pipeline-state.json'
                if state_file.exists():
                    state = json.loads(state_file.read_text())
                    print('Pipeline:', state, flush=True)
                    if 'stage' in state:
                        stage_log = WORK / (state['stage'] + '.log')
                        if stage_log.exists():
                            with stage_log.open('rb') as source:
                                source.seek(max(0, stage_log.stat().st_size - 1600))
                                print(source.read().decode(errors='replace'), flush=True)
                last_report = time.monotonic()
            time.sleep(5)
    except KeyboardInterrupt:
        pause.write_text(json.dumps({'reason': 'notebook user interrupted; checkpoint requested'}))
        print('Checkpoint pause requested. Keep this runtime open until the study stops.', flush=True)
        raise
if process.returncode:
    raise RuntimeError('Study launcher failed; inspect ' + str(supervisor_log))
print(json.loads((WORK / 'pipeline-state.json').read_text()))


In [ ]:
state = json.loads((WORK / 'pipeline-state.json').read_text())
if state['status'] != 'complete':
    raise RuntimeError('Full fine-tuning is not complete: ' + repr(state))
run_script('report_beats_adaptation.py', '--work', WORK, '--output', REPORT)
frozen = json.loads((WORK / 'frozen-controls.json').read_text())
adapted = json.loads((WORK / 'fine-tuned-controls.json').read_text())
table = []
for family, results in [('frozen', frozen), ('fine-tuned', adapted)]:
    for held_source, variants in results.items():
        for mix, result in variants.items():
            table.append({'encoder': family, 'held_source': held_source, 'positive_mix': mix,
                          'validation': result['validation']['by_source']})
print(json.dumps(table, indent=2))
print('Complete results and retained trained parameters:', WORK)
print('Verified comparison report:', REPORT / 'RESULTS.md')
print('These are held-source development comparisons, using unchanged weak publisher labels.')
